# 📄 Module 4.1 — PDF Splitting & Classification v5
**Bổ sung so với v4 (đối chiếu AI Requirements doc):**

| Task | Section | Nội dung |
|------|---------|----------|
| T1 | §5 | `classification` = nhãn thực; `summary` = mô tả văn bản |
| T2 | §3.1, §5 | `metadata[]` — bóc tách `so_van_ban`, `ngay_ky`, `co_quan`, `ten_loai_vb` |
| T3 | §4.2 | Auto-summary trích yếu (rule-based + Qwen fallback) |
| T4 | §6.4 | Error handling — mã lỗi chuẩn JSON |
| T5 | §6.1 | SLA per-page timing |


## ⚙️ Cell 1 — Cài đặt

In [ ]:
!pip install pymupdf pdf2image pytesseract tqdm requests -q
!apt-get install -y tesseract-ocr tesseract-ocr-vie poppler-utils -q 2>/dev/null

import fitz, re, json, uuid, time, os, pickle, hashlib, requests
import concurrent.futures
from pathlib import Path
from datetime import datetime
from PIL import Image
from tqdm.auto import tqdm
import pytesseract

print("✅ Setup xong!")

## ⚙️ Cell 2 — Config & Ground Truth

In [ ]:
from google.colab import drive
import os, shutil

mount_point = '/content/drive'
if os.path.exists(mount_point) and not os.path.ismount(mount_point) and os.listdir(mount_point):
    for item in os.listdir(mount_point):
        item_path = os.path.join(mount_point, item)
        try:
            if os.path.isfile(item_path) or os.path.islink(item_path): os.remove(item_path)
            elif os.path.isdir(item_path): shutil.rmtree(item_path)
        except Exception as e: print(f"Error clearing {item_path}: {e}")

drive.mount(mount_point, force_remount=True)
os.listdir(os.path.join(mount_point, 'MyDrive/VNDigitizeComprehensiveSystem_Team03/dataset_module4'))

PDF_PATH   = '/content/drive/MyDrive/VNDigitizeComprehensiveSystem_Team03/dataset_module4/mix_doc/document_merged2.pdf'
OUTPUT_DIR = '/content/drive/MyDrive/VNDigitizeComprehensiveSystem_Team03/dataset_module4/output'
CACHE_PATH = '/content/drive/MyDrive/VNDigitizeComprehensiveSystem_Team03/dataset_module4/cache_features.pkl'

CACHE_VERSION  = "5.0"
OCR_DPI        = 200
BLANK_THRESH   = 30
TEXT_MIN_CHARS = 20
OCR_WORKERS    = 4
MIN_SEG_PAGES  = 1

RE_SO_VAN_BAN = re.compile(r'\d{1,3}/\d{4,}/[A-ZĐQĐ][\w\-]+', re.IGNORECASE)

LABEL_SET = {"Quyết định", "Bản án dân sự", "Bản án hình sự", "Bản án hành chính"}

GT_BOUNDARIES = {1,4,7,9,29,35,43,44,50,52,54,57,59,62,63,66,75,81,103,105,108}
GT_SEPARATORS = {65}
GT_SEGMENT_TYPES = {
    1:"Quyết định",      4:"Quyết định",      7:"Quyết định",
    9:"Bản án dân sự",  29:"Quyết định",     35:"Bản án dân sự",
   43:"Bản án hình sự", 44:"Bản án hình sự", 50:"Quyết định",
   52:"Quyết định",     54:"Quyết định",     57:"Quyết định",
   59:"Quyết định",     62:"Quyết định",     63:"Quyết định",
   66:"Bản án hình sự", 75:"Bản án dân sự",  81:"Bản án hành chính",
  103:"Quyết định",    105:"Quyết định",    108:"Quyết định",
}

try:
    with fitz.open(PDF_PATH) as doc: _n = len(doc)
    print(f"✅ Config xong | {Path(PDF_PATH).name} | {_n} trang")
except Exception as e:
    print(f"⚠️  PDF_PATH chưa đúng: {e}")

## ⚡ Cell 3 — Fast Extract (2-pass · cache-safe · fork-safe)

In [ ]:
# ── Cache helpers ────────────────────────────────────────────
def _cache_key(pdf_path: str) -> str:
    p = Path(pdf_path)
    mtime = str(p.stat().st_mtime) if p.exists() else "0"
    return hashlib.md5(f"{pdf_path}|{mtime}|{CACHE_VERSION}".encode()).hexdigest()[:12]

def _load_cache(cache_path: str, pdf_path: str):
    if not Path(cache_path).exists(): return None
    try:
        with open(cache_path, "rb") as f: bundle = pickle.load(f)
        if bundle.get("key") != _cache_key(pdf_path):
            print("  ⚠️  Cache lỗi thời → extract lại"); return None
        return bundle["features"]
    except Exception: return None

def _save_cache(cache_path: str, pdf_path: str, features: list):
    Path(cache_path).parent.mkdir(parents=True, exist_ok=True)
    with open(cache_path, "wb") as f:
        pickle.dump({"key": _cache_key(pdf_path), "features": features}, f)
    print(f"  💾 Cache: {cache_path}")

# ── Pass 1: text layer ────────────────────────────────────────
def _extract_text_layer(pdf_path: str) -> list[tuple[int, str]]:
    with fitz.open(pdf_path) as doc:
        return [(i, doc[i].get_text("text").strip()) for i in range(len(doc))]

# ── Pass 2: OCR (fork-safe: render trước, worker chỉ chạy Tesseract) ──
def _render_scan_pages(pdf_path: str, scan_idxs: list, dpi: int) -> dict:
    pages_bytes = {}
    with fitz.open(pdf_path) as doc:
        for idx in tqdm(scan_idxs, desc="  Render pixmap", unit="pg"):
            pix = doc[idx].get_pixmap(dpi=dpi, colorspace=fitz.csGRAY)
            pages_bytes[idx] = (bytes(pix.samples), pix.width, pix.height)
    return pages_bytes

def _ocr_from_bytes(args: tuple) -> tuple[int, str, float]:
    """Worker fork-safe: chỉ nhận bytes, không dùng fitz."""
    idx, samples, w, h = args
    from PIL import Image; import pytesseract, time
    t0  = time.time()
    img = Image.frombytes("L", [w, h], samples)
    txt = pytesseract.image_to_string(img, lang='vie', config='--oem 1 --psm 3').strip()
    return idx, txt, time.time() - t0  # T5: trả thêm thời gian xử lý

# ── Feature builder ───────────────────────────────────────────
def _strict_header(text: str, n: int = 7) -> tuple[str, str]:
    lines, result, first = (text.split("\n") if text else []), [], ""
    for line in lines:
        s = line.strip()
        if not s or re.match(r'^[\d\-–—\s]{1,5}$', s): continue
        if not first: first = s
        result.append(s)
        if len(result) >= n: break
    return " ".join(result), first

def _build_feature(page_idx: int, text: str, elapsed_s: float = 0.0) -> dict:
    text2000   = text[:2000]
    header_raw = " ".join(l.strip() for l in text2000.split("\n")[:10] if l.strip())
    header_up  = header_raw.upper()
    sh, fl     = _strict_header(text)
    return {
        "page_num"      : page_idx + 1,
        "char_count"    : len(text),
        "is_blank"      : len(text) < BLANK_THRESH,
        "header"        : header_raw,
        "has_quoc_hieu" : "CỘNG HÒA XÃ HỘI" in header_up,
        "has_tieu_ngu"  : "HẠNH PHÚC"        in header_up,
        "has_toa_an"    : "TÒA ÁN NHÂN DÂN"  in header_up,
        "so_vb_header"  : RE_SO_VAN_BAN.findall(header_raw),
        "text_full"     : text,
        "strict_header" : (sh, fl),
        "extract_ms"    : round(elapsed_s * 1000, 1),  # T5: timing per page
    }

# ── Main extract ──────────────────────────────────────────────
def batch_extract(pdf_path: str, cache_path: str | None = None,
                  force: bool = False) -> list[dict]:
    if cache_path and not force:
        cached = _load_cache(cache_path, pdf_path)
        if cached is not None:
            print(f"✅ Load từ cache ({len(cached)} trang)"); return cached
    t0 = time.time()

    # Pass 1
    print("  [Pass 1] Text layer...")
    t1 = time.time(); raw = _extract_text_layer(pdf_path)
    print(f"  → {len(raw)} trang | {time.time()-t1:.1f}s")

    has_text, needs_ocr = [], []
    for idx, text in raw:
        (has_text if len(re.sub(r'\s+','',text)) >= TEXT_MIN_CHARS else needs_ocr).append((idx, text) if len(re.sub(r'\s+','',text)) >= TEXT_MIN_CHARS else idx)

    # Rebuild cleanly
    has_text_d = {idx: txt for idx, txt in raw if len(re.sub(r'\s+','',txt)) >= TEXT_MIN_CHARS}
    needs_ocr  = [idx for idx, txt in raw if len(re.sub(r'\s+','',txt)) < TEXT_MIN_CHARS]
    print(f"  → {len(has_text_d)} text-layer | {len(needs_ocr)} scan (OCR)")

    ocr_results: dict[int, tuple[str, float]] = {}
    if needs_ocr:
        print(f"  [Pass 2] Render {len(needs_ocr)} scan pages...")
        pages_bytes = _render_scan_pages(pdf_path, needs_ocr, OCR_DPI)
        t2  = time.time()
        args = [(idx, *pages_bytes[idx]) for idx in needs_ocr]
        with concurrent.futures.ProcessPoolExecutor(max_workers=OCR_WORKERS) as ex:
            for idx, txt, ms in tqdm(ex.map(_ocr_from_bytes, args),
                                     total=len(args), desc="  OCR", unit="pg"):
                ocr_results[idx] = (txt, ms)
        print(f"  → OCR xong: {time.time()-t2:.1f}s")

    features = []
    for i in range(len(raw)):
        if i in has_text_d:
            features.append(_build_feature(i, has_text_d[i], 0.0))
        else:
            txt, ms = ocr_results.get(i, ("", 0.0))
            features.append(_build_feature(i, txt, ms))

    print(f"  ✅ Extract: {time.time()-t0:.1f}s")
    if cache_path: _save_cache(cache_path, pdf_path, features)
    return features

print("✅ Fast Extract (v5) xong.")

## 🔍 Cell 4 — Boundary Detection + Merge segment ảo

In [ ]:
_FIRST_LINE_RE = re.compile(
    r'TÒA ÁN NHÂN DÂN|VIỆN KIỂM SÁT|CỘNG HÒA XÃ HỘI'
    r'|NHÂN DANH|QUYẾT ĐỊNH\s*$|ĐÌNH CHỈ|THÔNG BÁO'
)

def _is_doc_start(strict_hdr: str, first_line: str) -> tuple[bool, str]:
    h, fl = strict_hdr.upper(), first_line.upper()
    if not _FIRST_LINE_RE.search(fl): return False, ""
    grp_a = "CỘNG HÒA XÃ HỘI" in h or "ĐỘC LẬP" in h or "HẠNH PHÚC" in h
    grp_b = "TÒA ÁN NHÂN DÂN"  in h or "VIỆN KIỂM SÁT" in h
    grp_c = (bool(RE_SO_VAN_BAN.search(strict_hdr))
             or "BẢN ÁN SỐ" in h or "NHÂN DANH" in h
             or bool(re.search(r'QUYẾT ĐỊNH\s*$', h, re.M))
             or "ĐÌNH CHỈ VỤ ÁN" in h or "THÔNG BÁO THỤ LÝ" in h)
    n_groups = sum([grp_a, grp_b, grp_c])
    if n_groups >= (1 if (grp_a and "CỘNG HÒA" in h) else 2):
        sigs = (["quoc_hieu"] if grp_a else []) +                (["co_quan"]   if grp_b else []) +                (["dinh_danh"] if grp_c else [])
        return True, "+".join(sigs)
    return False, ""

def detect_boundary(feat, prev):
    if prev is None:     return True,  "first_page",  1.0
    if feat["is_blank"]: return False, "blank_page",  0.0
    if prev["is_blank"]: return True,  "after_blank", 0.95
    sh, fl = feat["strict_header"]
    is_s, sig = _is_doc_start(sh, fl)
    if is_s:
        return True, sig, 1.0 if ("quoc_hieu" in sig and "co_quan" in sig) else 0.9
    prev_vb, curr_vb = set(prev.get("so_vb_header",[])), set(feat.get("so_vb_header",[]))
    if curr_vb and prev_vb and not curr_vb & prev_vb: return True, "new_so_vb", 0.85
    return False, "continuation", 0.0

def compute_boundaries(feats):
    boundaries, separators = set(), set()
    for i, f in enumerate(feats):
        is_bd,_,_ = detect_boundary(f, feats[i-1] if i > 0 else None)
        if is_bd:         boundaries.add(f["page_num"])
        if f["is_blank"]: separators.add(f["page_num"])
    return boundaries, separators

def group_segments(feats, boundaries, separators):
    segs, cur = [], []
    for f in feats:
        pn = f["page_num"]
        if f["is_blank"] or pn in separators:
            if cur: segs.append(cur); cur = []
            continue
        if pn in boundaries and cur: segs.append(cur); cur = []
        cur.append(f)
    if cur: segs.append(cur)
    return segs

def _seg_has_clear_start(pages):
    if not pages: return False
    f  = pages[0]
    sh = f.get("strict_header", ("",""))[0].upper()
    return (f.get("has_quoc_hieu") or f.get("has_toa_an") or
            bool(f.get("so_vb_header")) or "NHÂN DANH" in sh or "BẢN ÁN SỐ" in sh)

def merge_short_segments(segs, min_pages=MIN_SEG_PAGES):
    if min_pages < 1 or not segs: return segs
    merged = [segs[0]]
    for seg in segs[1:]:
        if len(seg) <= min_pages and not _seg_has_clear_start(seg):
            merged[-1] = merged[-1] + seg
            print(f"  🔀 Merge p{seg[0]['page_num']} → seg p{merged[-1][0]['page_num']}-{merged[-1][-1]['page_num']}")
        else: merged.append(seg)
    return merged

print("✅ Boundary Detection v5 xong.")

## 🏷️ Cell 5 — Rule-Based Classifier

In [ ]:
_SPECIAL_CODES = {
    'QĐST-HC':"Quyết định",'QĐPT-HC':"Quyết định",'QĐ-PT':"Quyết định",'QĐ-ST':"Quyết định",
}
_KY_HIEU_MAP = [
    (re.compile(r'/QĐ[\-\w]*',                       re.I), "Quyết định"),
    (re.compile(r'/QĐST[\-\w]*',                     re.I), "Quyết định"),
    (re.compile(r'/QĐPT[\-\w]*',                     re.I), "Quyết định"),
    (re.compile(r'/[\w]*HC[\-]*(ST|PT|GĐT)\b',      re.I), "Bản án hành chính"),
    (re.compile(r'/[\w]*HS[\-]*(ST|PT|GĐT|SĐT)\b',  re.I), "Bản án hình sự"),
    (re.compile(r'/[\w]*(DS|HNGĐ|KDTM)[\-]*(ST|PT|GĐT)?\b', re.I), "Bản án dân sự"),
]
_KEYWORDS = [
    ("BẢN ÁN HÌNH SỰ","Bản án hình sự",5),("VỀ TỘI","Bản án hình sự",3),
    ("BỊ CÁO","Bản án hình sự",4),("PHẠM TỘI","Bản án hình sự",3),
    ("BẢN ÁN DÂN SỰ","Bản án dân sự",5),("BẢN ÁN HÔN NHÂN","Bản án dân sự",5),
    ("LY HÔN","Bản án dân sự",4),("TRANH CHẤP HỢP ĐỒNG","Bản án dân sự",3),
    ("NGUYÊN ĐƠN","Bản án dân sự",2),("BỊ ĐƠN","Bản án dân sự",2),
    ("BẢN ÁN HÀNH CHÍNH","Bản án hành chính",5),
    ("KHỞI KIỆN QUYẾT ĐỊNH HÀNH CHÍNH","Bản án hành chính",5),
    ("NGƯỜI BỊ KIỆN","Bản án hành chính",3),
]
_HARD_QD = re.compile(r'QUYẾT ĐỊNH\s*(ĐÌNH CHỈ|V/V|SỐ\s*\d|XÉT XỬ|CÔNG NHẬN)', re.I)
_DINH_CHI = ["ĐÌNH CHỈ VỤ ÁN","ĐÌNH CHỈ XÉT XỬ","ĐÌNH CHỈ PHIÊN TÒA"]

def _by_so_vb(so_vb):
    su = so_vb.upper()
    for code, lbl in _SPECIAL_CODES.items():
        if code in su: return lbl
    for pat, lbl in _KY_HIEU_MAP:
        if pat.search(so_vb): return lbl
    return None

def _is_hard_qd(head):
    h = head.upper()
    return _HARD_QD.search(h) is not None or any(p in h for p in _DINH_CHI)

def _by_keywords(text):
    t2000, t500 = text[:2000].upper(), text[:500].upper()
    scores = dict.fromkeys(LABEL_SET, 0)
    is_qd = bool(re.search(r'QUYẾT ĐỊNH\s*[\n\r]', t500)) or _is_hard_qd(t500)
    if is_qd:
        scores["Quyết định"] += 5
        for kw, lbl, w in _KEYWORDS:
            if lbl not in ("Bản án hình sự","Bản án dân sự") and kw in t2000:
                scores[lbl] += w
    else:
        if re.search(r'QUYẾT ĐỊNH\s{0,5}(V/V|SỐ|ĐÌNH CHỈ)', t500): scores["Quyết định"] += 5
        for kw, lbl, w in _KEYWORDS:
            if kw in t2000: scores[lbl] += w
    best = max(scores, key=scores.get)
    if scores[best] == 0: return None, 0.0
    vals = sorted(scores.values(), reverse=True)
    return best, min(0.95, 0.6 + (vals[0]-vals[1] if len(vals)>1 else vals[0]) * 0.05)

def classify_rule(pages):
    texts = [p["text_full"] for p in pages[:3]]
    if texts and _is_hard_qd(texts[0][:500]): return "Quyết định", 0.97
    for t in texts:
        for sv in RE_SO_VAN_BAN.findall(t):
            lbl = _by_so_vb(sv)
            if lbl: return lbl, 0.98
    lbl, conf = _by_keywords(" ".join(texts))
    if lbl: return lbl, conf
    if len(pages) > 1:
        t2 = pages[1]["text_full"]
        for sv in RE_SO_VAN_BAN.findall(t2):
            lbl = _by_so_vb(sv); 
            if lbl: return lbl, 0.90
        lbl2, conf2 = _by_keywords(t2)
        if lbl2: return lbl2, max(0.5, conf2-0.1)
    return "Quyết định", 0.50

print("✅ Classifier v5 xong.")

## 📋 Cell 6 — Metadata Extraction (T1 + T2: §3.1, §5)

In [ ]:
# ══════════════════════════════════════════════════════════════
# TASK 1 + 2: Bóc tách metadata[] theo JSON schema yêu cầu
#
# Trường cần extract:
#   so_van_ban      → RE_SO_VAN_BAN đã có
#   ngay_ky         → regex ngày tháng năm
#   co_quan_ban_hanh → dòng sau quốc hiệu / trước số văn bản
#   ten_loai_vb     → = classification label
# ══════════════════════════════════════════════════════════════

# Regex ngày: "ngày 12 tháng 3 năm 2024" hoặc "12/03/2024" hoặc "12-03-2024"
RE_NGAY = re.compile(
    r'ngày\s*(\d{1,2})\s*tháng\s*(\d{1,2})\s*năm\s*(\d{4})'
    r'|\b(\d{1,2})[/\-](\d{1,2})[/\-](\d{4})\b',
    re.IGNORECASE
)

# Regex cơ quan ban hành: dòng CÓ "TÒA ÁN" / "UBND" / "BỘ" / "CỤC" / "SỞ" / "VIỆN"
RE_CO_QUAN = re.compile(
    r'(TÒA ÁN[\w\s]+|UBND[\w\s,]+|BỘ[\w\s]+|'
    r'CỤC[\w\s]+|SỞ[\w\s]+|VIỆN KIỂM SÁT[\w\s]+|'
    r'PHÒNG[\w\s]+)',
    re.IGNORECASE
)

def _norm_ngay(m: re.Match) -> str:
    """Chuẩn hóa ngày từ regex match → 'DD/MM/YYYY'."""    g = m.groups()
    if g[0]:   # dạng chữ: ngày X tháng Y năm Z
        return f"{g[0].zfill(2)}/{g[1].zfill(2)}/{g[2]}"
    else:      # dạng số: DD/MM/YYYY
        return f"{g[3].zfill(2)}/{g[4].zfill(2)}/{g[5]}"

def extract_metadata(pages: list[dict], label: str, conf: float) -> list[dict]:
    """
    T2: Bóc tách các trường metadata từ 2 trang đầu của segment.
    Trả về list[dict] đúng JSON schema §5.
    """
    # Gộp text 2 trang đầu (đủ để tìm header fields)
    head_text = " ".join(p["text_full"] for p in pages[:2])[:3000]

    metadata = []

    # ── ten_loai_vb ──────────────────────────────────────────
    metadata.append({
        "field_name": "ten_loai_vb",
        "value"     : label,
        "confidence": round(conf, 3),
        "bounding_box": None,
    })

    # ── so_van_ban ───────────────────────────────────────────
    so_vb_list = RE_SO_VAN_BAN.findall(head_text)
    if so_vb_list:
        metadata.append({
            "field_name": "so_van_ban",
            "value"     : so_vb_list[0].upper(),
            "confidence": 0.97,
            "bounding_box": None,
        })

    # ── ngay_ky ───────────────────────────────────────────────
    ngay_m = RE_NGAY.search(head_text)
    if ngay_m:
        metadata.append({
            "field_name": "ngay_ky",
            "value"     : _norm_ngay(ngay_m),
            "confidence": 0.92,
            "bounding_box": None,
        })

    # ── co_quan_ban_hanh ──────────────────────────────────────
    # Ưu tiên: dòng "TÒA ÁN NHÂN DÂN ..." / "UBND ..." trong header
    co_quan_val = None
    for line in head_text.split("\n"):
        ls = line.strip()
        if len(ls) < 5: continue
        cq_m = RE_CO_QUAN.match(ls)
        if cq_m:
            co_quan_val = ls[:80].strip()
            break
    # Fallback: tìm trong toàn bộ head
    if not co_quan_val:
        cq_m2 = RE_CO_QUAN.search(head_text[:500])
        if cq_m2: co_quan_val = cq_m2.group(0)[:80].strip()
    if co_quan_val:
        metadata.append({
            "field_name": "co_quan_ban_hanh",
            "value"     : co_quan_val,
            "confidence": 0.82,
            "bounding_box": None,
        })

    return metadata

print("✅ Metadata Extractor (v5) xong.")

## 🤖 Cell 7 — Qwen2.5 (Classify + Auto-summary)

In [ ]:
QWEN_URL     = "http://localhost:11434/api/generate"
QWEN_MODEL   = "qwen2.5:7b"
QWEN_TIMEOUT = 30

_QWEN_CLASSIFY_PROMPT = (
    "Bạn là chuyên gia phân loại tài liệu pháp lý Việt Nam.\n"
    "Chỉ trả lời đúng 1 trong 4 nhãn (không giải thích):\n"
    "  Quyết định\n  Bản án dân sự\n  Bản án hình sự\n  Bản án hành chính"
)

# T3: prompt sinh trích yếu
_QWEN_SUMMARY_PROMPT = (
    "Bạn là chuyên gia tóm tắt văn bản pháp lý Việt Nam.\n"
    "Đọc đoạn văn bản sau và viết 1 câu trích yếu ngắn gọn (tối đa 30 từ), "
    "nêu rõ: loại văn bản, số hiệu (nếu có), cơ quan ban hành, nội dung chính.\n"
    "Chỉ trả lời 1 câu, không giải thích thêm."
)

def _qwen_classify(text_head: str) -> tuple[str | None, float]:
    prompt = f"{_QWEN_CLASSIFY_PROMPT}\n\n---\n{text_head[:800]}\n---\nNhãn:"
    try:
        resp = requests.post(QWEN_URL, json={"model": QWEN_MODEL, "prompt": prompt,
                  "stream": False, "options": {"temperature": 0}}, timeout=QWEN_TIMEOUT)
        if resp.status_code != 200: return None, 0.0
        raw = resp.json().get("response", "").strip()
        for lbl in LABEL_SET:
            if lbl.lower() in raw.lower(): return lbl, 0.88
        return None, 0.0
    except Exception: return None, 0.0

def _qwen_summary(text_head: str) -> str | None:
    """T3: Sinh 1 câu trích yếu bằng Qwen."""    prompt = f"{_QWEN_SUMMARY_PROMPT}\n\n---\n{text_head[:1200]}\n---\nTrích yếu:"
    try:
        resp = requests.post(QWEN_URL, json={"model": QWEN_MODEL, "prompt": prompt,
                  "stream": False, "options": {"temperature": 0.3, "num_predict": 80}},
                  timeout=QWEN_TIMEOUT)
        if resp.status_code != 200: return None
        return resp.json().get("response", "").strip() or None
    except Exception: return None

def _rule_summary(meta: list[dict], label: str) -> str:
    """T3: Fallback rule-based nếu Qwen offline."""    fields = {m["field_name"]: m["value"] for m in meta}
    so_vb  = fields.get("so_van_ban", "")
    ngay   = fields.get("ngay_ky", "")
    cq     = fields.get("co_quan_ban_hanh", "")
    parts  = [label]
    if so_vb: parts.append(f"số {so_vb}")
    if ngay:  parts.append(f"ngày {ngay}")
    if cq:    parts.append(f"của {cq}")
    return " ".join(parts) + "." if len(parts) > 1 else f"{label} (chưa trích được thông tin)."

def _qwen_available() -> bool:
    try: return requests.get("http://localhost:11434", timeout=3).status_code == 200
    except Exception: return False

QWEN_ON = _qwen_available()
print(f"🤖 Qwen2.5 = {'✅ online' if QWEN_ON else '⚠️  offline – rule-based fallback'}")

## 🚨 Cell 8 — Mislabel Detector

In [ ]:
def detect_mislabels(segments: list[dict], gt: dict | None = None) -> list[dict]:
    flagged = []
    for seg in segments:
        ps, rule, qwen, conf = (seg["page_start"], seg.get("type"),
                                 seg.get("qwen_type"), seg.get("confidence", 1.0))
        gt_lbl  = gt.get(ps) if gt else None
        reasons = []
        if qwen and qwen in LABEL_SET and rule != qwen:
            reasons.append(f"[{'HIGH' if conf>=0.95 else 'MED'}] rule={rule} ≠ qwen={qwen}")
        if conf < 0.70:
            reasons.append(f"low_conf={conf:.2f}")
        if gt_lbl and gt_lbl != rule:
            reasons.append(f"gt={gt_lbl} ≠ pred={rule}")
        if reasons:
            flagged.append({
                "segment": seg.get("segment"), "pages": f"p{seg['page_start']}-{seg['page_end']}",
                "rule_label": rule, "qwen_label": qwen, "gt_label": gt_lbl,
                "confidence": conf, "reasons": reasons,
                "suggested": gt_lbl or (qwen if qwen in LABEL_SET else rule),
            })
    return flagged

def print_mislabel_report(flagged: list[dict]):
    if not flagged: print("✅ Không phát hiện nhãn sai!"); return
    high = [f for f in flagged if any("HIGH" in r for r in f["reasons"])]
    print(f"\n🚨 {len(flagged)} flag | {len(high)} HIGH severity")
    print("-" * 76)
    for f in flagged:
        icon = "🔴" if any("HIGH" in r for r in f["reasons"]) else "🟡"
        print(f"  {icon} Seg {f['segment']:>3} | {f['pages']:>12} | rule={f['rule_label']:<22} qwen={str(f['qwen_label']):<20}")
        for r in f["reasons"]: print(f"         · {r}")
        print(f"         ➜ gợi ý: {f['suggested']}")
    print("-" * 76)

print("✅ Mislabel Detector v5 xong.")

## 🛡️ Cell 9 — Error Handling (T4: §6.4)

In [ ]:
# ══════════════════════════════════════════════════════════════
# TASK 4: Error codes chuẩn theo §6.4 AI Requirements
#
# ERR_PDF_ENCRYPTED  → tệp PDF có mật khẩu
# ERR_FILE_NOT_FOUND → đường dẫn không tồn tại
# ERR_FILE_CORRUPTED → fitz không mở được / bị hỏng
# ERR_BLANK_DOC      → toàn bộ trang đều blank
# ERR_UNKNOWN        → lỗi không xác định
# ══════════════════════════════════════════════════════════════

class PipelineError(Exception):
    """Lỗi có thể chuyển thành JSON error response."""    def __init__(self, code: str, message: str):
        self.code = code; self.message = message
        super().__init__(message)

def _check_pdf(pdf_path: str) -> int:
    """Kiểm tra PDF trước khi xử lý. Raise PipelineError nếu có vấn đề."""
    if not Path(pdf_path).exists():
        raise PipelineError("ERR_FILE_NOT_FOUND", f"File không tồn tại: {pdf_path}")
    try:
        with fitz.open(pdf_path) as doc:
            if doc.is_encrypted:
                raise PipelineError("ERR_PDF_ENCRYPTED",
                    "Tệp PDF được bảo vệ bằng mật khẩu. Vui lòng cung cấp mật khẩu.")
            n = len(doc)
            if n == 0:
                raise PipelineError("ERR_BLANK_DOC", "Tệp PDF không có trang nào.")
            return n
    except PipelineError: raise
    except Exception as e:
        raise PipelineError("ERR_FILE_CORRUPTED", f"Không thể đọc tệp PDF: {e}")

def _error_response(code: str, message: str, pdf_path: str) -> dict:
    """Trả về JSON error chuẩn thay vì raise exception."""    return {
        "document_id"    : str(uuid.uuid4())[:8],
        "error_code"     : code,
        "error_message"  : message,
        "pdf_path"       : pdf_path,
        "processing_ms"  : 0,
        "sub_documents"  : [],
        "metadata"       : [],
    }

def _check_blank_result(feats: list) -> None:
    """Nếu toàn bộ trang đều blank sau extract → raise."""    non_blank = [f for f in feats if not f["is_blank"]]
    if not non_blank:
        raise PipelineError("ERR_BLANK_DOC",
            "Tất cả các trang đều trắng hoặc không đọc được nội dung.")

print("✅ Error Handling v5 xong.")

## 🚀 Cell 10 — Full Pipeline v5 (tích hợp T1–T5)

In [ ]:
def full_pipeline(
    pdf_path         : str,
    output_dir       : str,
    cache_path       : str | None = None,
    gt_boundaries    : set  | None = None,
    gt_segment_types : dict | None = None,
    use_qwen         : bool = True,
    force_extract    : bool = False,
    use_timestamp_dir: bool = True,
) -> dict:
    # T4: kiểm tra file trước, trả error JSON nếu có vấn đề
    try:
        _check_pdf(pdf_path)
    except PipelineError as e:
        print(f"❌ {e.code}: {e.message}")
        return _error_response(e.code, e.message, pdf_path)

    t0 = time.time()
    run_dir = Path(output_dir) / (datetime.now().strftime("%Y%m%d_%H%M%S")
                                   if use_timestamp_dir else "")
    run_dir.mkdir(parents=True, exist_ok=True)

    # ── 1. Extract ────────────────────────────────────────────
    print("📑 [1/5] Extracting text...")
    try:
        feats = batch_extract(pdf_path, cache_path=cache_path, force=force_extract)
        _check_blank_result(feats)  # T4: kiểm tra blank
    except PipelineError as e:
        return _error_response(e.code, e.message, pdf_path)

    total_pages  = len(feats)
    # T5: tổng hợp timing per-page từ extract_ms trong features
    page_timing  = {f["page_num"]: f.get("extract_ms", 0.0) for f in feats}

    # ── 2. Boundary ───────────────────────────────────────────
    print("🔍 [2/5] Detecting boundaries...")
    boundaries, separators = compute_boundaries(feats)
    raw_segs = group_segments(feats, boundaries, separators)
    segs     = merge_short_segments(raw_segs)
    print(f"   → {len(raw_segs)} raw → {len(segs)} sau merge")

    # ── 3. Classify + Metadata + Summary ─────────────────────
    print("🏷️  [3/5] Classifying + extracting metadata...")
    results = []
    for i, pages in enumerate(segs):
        rule_lbl, rule_conf = classify_rule(pages)

        qwen_lbl = None
        if use_qwen and QWEN_ON:
            head     = " ".join(p["text_full"] for p in pages[:2])[:1200]
            qwen_lbl, _ = _qwen_classify(head)

        if qwen_lbl and qwen_lbl in LABEL_SET and rule_conf < 0.80:
            final_lbl, final_conf = qwen_lbl, 0.88
        else:
            final_lbl, final_conf = rule_lbl, rule_conf

        # T2: extract metadata fields
        meta = extract_metadata(pages, final_lbl, final_conf)

        # T3: auto-summary
        if use_qwen and QWEN_ON:
            head_txt = " ".join(p["text_full"] for p in pages[:2])[:1200]
            summary  = _qwen_summary(head_txt) or _rule_summary(meta, final_lbl)
        else:
            summary = _rule_summary(meta, final_lbl)

        results.append({
            "segment"   : i + 1,
            "page_start": pages[0]["page_num"],
            "page_end"  : pages[-1]["page_num"],
            "n_pages"   : len(pages),
            "type"      : final_lbl,
            "rule_type" : rule_lbl,
            "qwen_type" : qwen_lbl,
            "confidence": round(final_conf, 3),
            "metadata"  : meta,       # T2
            "summary"   : summary,    # T3
        })

    # ── 4. Mislabel ───────────────────────────────────────────
    print("🚨 [4/5] Checking mislabels...")
    flagged = detect_mislabels(results, gt=gt_segment_types)
    print_mislabel_report(flagged)

    # ── 5. Split & Save ───────────────────────────────────────
    print("✂️  [5/5] Splitting PDFs...")
    src      = fitz.open(pdf_path)
    doc_id   = str(uuid.uuid4())[:8]
    sub_docs = []

    for r in results:
        lbl_safe = r["type"].replace(" ", "_")
        fname    = f"sub_{r['segment']:03d}_{lbl_safe}_p{r['page_start']}-{r['page_end']}.pdf"
        out_pdf  = fitz.open()
        out_pdf.insert_pdf(src, from_page=r["page_start"]-1, to_page=r["page_end"]-1)
        out_pdf.save(str(run_dir / fname))
        out_pdf.close()
        sub_docs.append({
            "type"      : r["type"],
            "page_start": r["page_start"],
            "page_end"  : r["page_end"],
        })
    src.close()

    elapsed  = int((time.time() - t0) * 1000)
    avg_conf = round(sum(r["confidence"] for r in results) / max(len(results), 1), 3)

    # T1: classification = nhãn thực (không hardcode)
    labels_in_doc = list({r["type"] for r in results})
    classification = labels_in_doc[0] if len(labels_in_doc) == 1 else "Tài liệu hỗn hợp"

    # JSON output đáp ứng đầy đủ schema §5
    payload = {
        "document_id"        : doc_id,
        "classification"     : classification,               # T1: nhãn thực
        "summary"            : results[0]["summary"] if results else "",  # T1+T3: summary
        "confidence_overall" : avg_conf,
        "processing_ms"      : elapsed,
        # T5: SLA per-page (top 10 trang chậm nhất)
        "slowest_pages_ms"   : sorted(
            [{"page": pn, "ms": ms} for pn, ms in page_timing.items() if ms > 0],
            key=lambda x: x["ms"], reverse=True
        )[:10],
        "mislabel_flags"     : len(flagged),
        "output_dir"         : str(run_dir),
        "metadata"           : results[0]["metadata"] if results else [],  # T2: từ seg đầu
        "sub_documents"      : sub_docs,
        # Extended: per-segment details (dùng nội bộ / debug)
        "_segments_detail"   : [
            {"segment": r["segment"], "pages": f"p{r['page_start']}-{r['page_end']}",
             "type": r["type"], "confidence": r["confidence"],
             "summary": r["summary"], "metadata": r["metadata"]}
            for r in results
        ],
    }

    json_path = run_dir / f"result_{doc_id}.json"
    with open(json_path, "w", encoding="utf-8") as fj:
        json.dump(payload, fj, ensure_ascii=False, indent=2)

    print(f"\n✅ Hoàn tất | {len(sub_docs)} segs | conf={avg_conf} | {elapsed}ms")
    print(f"   Output: {run_dir}")
    return payload

print("✅ Full Pipeline v5 xong.")

## ▶️ Cell 11 — Chạy

In [ ]:
result = full_pipeline(
    pdf_path         = PDF_PATH,
    output_dir       = OUTPUT_DIR,
    cache_path       = CACHE_PATH,
    gt_boundaries    = GT_BOUNDARIES,
    gt_segment_types = GT_SEGMENT_TYPES,
    use_qwen         = True,
    force_extract    = False,
    use_timestamp_dir= True,
)

# In JSON output
print(json.dumps(result, ensure_ascii=False, indent=2))

## 📊 Cell 12 — Đánh giá Accuracy + Metadata Quality

In [ ]:
def evaluate(result: dict, gt_types: dict) -> float:
    sub_docs = result.get("sub_documents", [])
    matched = total = 0
    print(f"\n{'Seg':>4} {'Pages':>12} {'Predict':>22} {'GT':>22} {'OK?':>5}")
    print("-" * 70)
    for i, s in enumerate(sub_docs, 1):
        gt = gt_types.get(s["page_start"])
        if gt is None: continue
        ok = s["type"] == gt; matched += ok; total += 1
        print(f"{i:>4} p{s['page_start']}-{s['page_end']:>3}  "
              f"{s['type']:>22}  {gt:>22}  {'✅' if ok else '❌'}")
    acc = matched / total * 100 if total else 0
    print(f"\nAccuracy: {matched}/{total} = {acc:.1f}%")

    # Kiểm tra metadata quality
    print("\n📋 Metadata coverage per segment:")
    for seg in result.get("_segments_detail", []):
        fields = [m["field_name"] for m in seg.get("metadata", [])]
        print(f"  Seg {seg['segment']:>3} {seg['pages']:<14} fields={fields}")
        print(f"         summary: {seg.get('summary','')[:80]}")
    return acc

if 'result' in dir() and 'error_code' not in result:
    evaluate(result, GT_SEGMENT_TYPES)